In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from app.ml.data.generate_sales_data import SalesDataGenerator
from app.ml.forecasting.prophet_forecaster import ProphetForecaster
from app.ml.forecasting.xgboost_forecaster import XGBoostForecaster
from app.ml.forecasting.ensemble import EnsembleForecaster
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

In [ ]:
# Generate synthetic multi-product data for evaluation
generator = SalesDataGenerator(random_seed=123)
sales_df = generator.generate_multi_product(days=540, num_products=5)
sales_df['ds'] = pd.to_datetime(sales_df['date'])
sales_df['y'] = sales_df['sales']
sales_df.head()

In [ ]:
def diebold_mariano(e1, e2, h=1, loss='mse'):
    d = (e1**2 - e2**2) if loss == 'mse' else (np.abs(e1) - np.abs(e2))
    d_mean = np.mean(d)
    gamma = np.array([np.cov(d[:-lag], d[lag:])[0,1] if lag != 0 else np.var(d, ddof=1) for lag in range(h)])
    var_d = gamma[0] + 2 * np.sum(gamma[1:])
    dm_stat = d_mean / np.sqrt(var_d / len(d))
    p_value = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    return dm_stat, p_value

In [ ]:
results = []
for product_id, group in sales_df.groupby('product_id'):
    history = group[['ds', 'y']].sort_values('ds').reset_index(drop=True)
    train = history.iloc[:-90]  # train on first 450 days
    test = history.iloc[-90:]

    # Prophet pipeline
    prophet = ProphetForecaster(model_dir='models')
    prophet.train(train, product_id=product_id)
    prophet_pred = prophet.predict(days_ahead=90, include_history=False)['yhat'].values

    # XGBoost pipeline
    xgb = XGBoostForecaster(model_dir='models')
    xgb_features = xgb.create_features(history)
    x_train = xgb_features.iloc[:-90].drop(columns=['ds', 'y'])
    y_train = xgb_features.iloc[:-90]['y']
    x_test = xgb_features.iloc[-90:].drop(columns=['ds', 'y'])
    y_test = xgb_features.iloc[-90:]['y']
    xgb.train(x_train, y_train, product_id=product_id)
    xgb_pred = xgb.predict(x_test)

    # Ensemble
    ensemble = EnsembleForecaster()
    weights = ensemble.adaptive_weighting([{
        'prophet': {'mape': np.mean(np.abs((y_test.values - prophet_pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100, 'rmse': np.sqrt(np.mean((y_test.values - prophet_pred)**2))},
        'xgboost': {'mape': np.mean(np.abs((y_test.values - xgb_pred) / np.where(y_test.values == 0, 1, y_test.values))) * 100, 'rmse': np.sqrt(np.mean((y_test.values - xgb_pred)**2))}
    }])
    ensemble_pred = ensemble.combine_predictions(prophet_pred, xgb_pred, weights=weights)

    # Metrics
    def metrics(y, yhat):
        rmse = np.sqrt(np.mean((y - yhat)**2))
        mae = np.mean(np.abs(y - yhat))
        mape = np.mean(np.abs((y - yhat) / np.where(y == 0, 1, y))) * 100
        return {'rmse': rmse, 'mae': mae, 'mape': mape}

    metrics_prophet = metrics(y_test.values, prophet_pred)
    metrics_xgb = metrics(y_test.values, xgb_pred)
    metrics_ens = metrics(y_test.values, ensemble_pred)

    dm_stat, dm_p = diebold_mariano(y_test.values - prophet_pred, y_test.values - xgb_pred, h=1)

    results.append({
        'product_id': product_id,
        'prophet': metrics_prophet,
        'xgboost': metrics_xgb,
        'ensemble': metrics_ens,
        'dm_stat': dm_stat,
        'dm_p': dm_p,
        'weights': weights
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
for r in results:
    print('Product:', r['product_id'])
    print('Prophet', r['prophet'])
    print('XGBoost', r['xgboost'])
    print('Ensemble', r['ensemble'])
    print('DM test p-value:', r['dm_p'])
    print('---')

In [ ]:
# Residual plots for one product
prod = results[0]['product_id']
history = sales_df[sales_df['product_id']==prod].sort_values('ds')
test = history.iloc[-90:]
# assuming preds from previous loops in arrays
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(test['ds'], test['y'], label='Actual')
ax.plot(test['ds'], prophet_pred, label='Prophet')
ax.plot(test['ds'], xgb_pred, label='XGBoost')
ax.plot(test['ds'], ensemble_pred, label='Ensemble', linestyle='--')
ax.set_title(f'Forecast vs Actual for {prod}')
ax.legend()
plt.show()